# Módulo 3 · Avance 2 — Auditoría de contratos con middleware real (HITL)

**Equipo:** Rebeca Elisa Guerra · Juan Carlos Huanca · David Rosendo Azcui

Este notebook conserva la base técnica y funcional del **Avance 1** (clave de Groq, `ChatGroq`, modelo `openai/gpt-oss-120b`, CUCE ficticios y herramientas de consulta/comparación/alerta) y corrige el control humano que en el Avance 2 estaba representado por una variable booleana propia.

## Qué demuestra este avance

1. **Acierto:** consulta un CUCE existente y compara el monto con las herramientas reales del agente.
2. **Error controlado:** consulta un CUCE inexistente y no continúa con la comparación.
3. **Human-in-the-loop real:** `HumanInTheLoopMiddleware` pausa el grafo antes de registrar una alerta; el operador reanuda con `approve` o `reject` mediante `Command(resume=...)`.
4. **Límites reales:** `ModelCallLimitMiddleware` limita llamadas al LLM y `ToolCallLimitMiddleware` limita ejecuciones de herramientas. Un caso aparte fuerza un presupuesto de una herramienta para observar el bloqueo.
5. **Evaluación objetiva:** tabla con `debe_usar`, `no_debe_usar` y porcentaje de aciertos, calculados sobre herramientas efectivamente ejecutadas y controles observados.

> **Importante sobre la credencial:** el Avance 1 usa **Groq** (`GROQ_API_KEY` y `ChatGroq`). No usa Grok de xAI (`XAI_API_KEY`). Mantengo Groq para conservar la integración original.

> **Alcance y cautela:** los dos CUCE, entidades y montos son ficticios. No hay conexión con SICOES ni verificación jurídica normativa. La acción de registro escribe una línea persistente en un archivo local append-only; el agente no dispone de herramientas para editarla o borrarla. Es una acción con efecto lateral real en el entorno del notebook, pero no equivale a una operación oficial ni a una irreversibilidad física/externa.

**Ejecución recomendada:** ejecutar las celdas en orden. Se requiere conexión a internet y una API key válida de Groq para ejecutar el agente y las pruebas; la tabla no inventa resultados si el modelo no sigue el contrato de herramientas.

## 0 · Instalación del entorno

Se actualizan LangChain y LangGraph a una versión que expone los middlewares nativos. Se mantiene `langchain-groq` para el proveedor usado en el Avance 1.

In [ ]:
%pip install -qU "langchain>=1.4.0" "langgraph>=1.2.0" langchain-groq pandas

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.


### 0.1 · Solicitud segura de la API key de Groq

En Google Colab se intenta leer primero **Secretos → `GROQ_API_KEY`**. Si no está disponible, se solicita con `getpass`, de modo que el valor no aparece en pantalla ni queda escrito en el notebook.

In [ ]:
import os
from getpass import getpass


def cargar_api_key_groq() -> None:
    """Carga GROQ_API_KEY desde Colab, entorno o entrada oculta."""
    clave = None
    try:
        from google.colab import userdata
        clave = userdata.get("GROQ_API_KEY")
        if clave:
            print("API key cargada desde Secretos de Colab.")
    except Exception:
        pass

    if not clave:
        clave = os.environ.get("GROQ_API_KEY")
        if clave:
            print("API key encontrada en variables de entorno.")

    if not clave:
        clave = getpass("Pega tu API key de Groq (GROQ_API_KEY): ").strip()

    if not clave:
        raise ValueError("No se recibió GROQ_API_KEY. Configura el secreto o vuelve a ejecutar esta celda.")

    os.environ["GROQ_API_KEY"] = clave
    print("GROQ_API_KEY disponible para esta sesión; no se muestra su contenido.")


cargar_api_key_groq()

API key cargada desde Secretos de Colab.
GROQ_API_KEY disponible para esta sesión; no se muestra su contenido.


### 0.2 · Modelo y limitador de frecuencia

Se conserva el modelo del Avance 1. El `InMemoryRateLimiter` controla la frecuencia de solicitudes; no reemplaza el límite lógico de llamadas que se configura más adelante con middleware.

In [ ]:
import time
from langchain_groq import ChatGroq
from langchain_core.rate_limiters import InMemoryRateLimiter

MODELO = "openai/gpt-oss-120b"
SOLICITUDES_POR_MINUTO = 20

limitador_frecuencia = InMemoryRateLimiter(
    requests_per_second=SOLICITUDES_POR_MINUTO / 60,
    check_every_n_seconds=0.1,
    max_bucket_size=2,
)

llm = ChatGroq(
    model=MODELO,
    temperature=0,
    reasoning_effort="low",
    max_retries=2,
    rate_limiter=limitador_frecuencia,
)

inicio = time.perf_counter()
prueba_llm = llm.invoke("Responde únicamente con la palabra: listo")
print(f"Proveedor: Groq | Modelo: {MODELO} | Respuesta: {prueba_llm.text!r}")
print(f"Latencia de comprobación: {time.perf_counter() - inicio:.2f} s")

Proveedor: Groq | Modelo: openai/gpt-oss-120b | Respuesta: 'listo'
Latencia de comprobación: 3.26 s


## 1 · Datos de demostración y primera definición de herramientas

Se retienen los CUCE y montos del Avance 1. Son **registros ficticios**. La verificación de montos es matemática y documental; no interpreta normas ni decide si un contrato es legal.

In [ ]:
import json
import os
import uuid
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import display
from langchain_core.tools import tool
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage

REGISTROS = {
    "2026-0451-00": {
        "cuce": "2026-0451-00",
        "entidad": "Ministerio de Obras Públicas (SIMULADO)",
        "monto_adjudicado": 150000.0,
        "estado": "ADJUDICADO",
        "moneda": "BOB",
    },
    "2026-0089-00": {
        "cuce": "2026-0089-00",
        "entidad": "Gobierno Autónomo Municipal (SIMULADO)",
        "monto_adjudicado": 85000.0,
        "estado": "CONTRATADO",
        "moneda": "BOB",
    },
    "2026-0312-05": {
        "cuce": "2026-0312-05",
        "entidad": "YPFB Corporación (SIMULADO)",
        "monto_adjudicado": 1200000.0,
        "estado": "ADJUDICADO",
        "moneda": "BOB",
    },
    "2026-0789-02": {
        "cuce": "2026-0789-02",
        "entidad": "Ministerio de Salud y Deportes (SIMULADO)",
        "monto_adjudicado": 450500.50,
        "estado": "EN_EJECUCION",
        "moneda": "BOB",
    },
    "2026-0999-10": {
        "cuce": "2026-0999-10",
        "entidad": "Gobierno Autónomo Regional (SIMULADO)",
        "monto_adjudicado": 0.0,
        "estado": "ANULADO",
        "moneda": "BOB",
    },
}

print("CUCE ficticios actualizados y disponibles:", list(REGISTROS))

# Bitácora en memoria para evaluar qué funciones llegaron a ejecutarse.
EVENTOS_HERRAMIENTAS: list[dict[str, Any]] = []

# Archivo persistente local. La herramienta solo añade registros; no expone delete/update.
ARCHIVO_ALERTAS = Path("alertas_aprobadas_append_only.jsonl")


def consultar_contrato_impl(cuce: str) -> dict[str, Any]:
    registro = REGISTROS.get(cuce)
    if registro is None:
        return {
            "ok": False,
            "error": "NO_ENCONTRADO",
            "cuce": cuce,
            "fuente": "REGISTROS_SIMULADOS",
        }
    return {"ok": True, "registro": registro, "fuente": "REGISTROS_SIMULADOS"}


@tool
def consultar_contrato(cuce: str) -> dict:
    """Consulta un CUCE en los dos registros contractuales ficticios de demostración."""
    resultado = consultar_contrato_impl(cuce)
    EVENTOS_HERRAMIENTAS.append({"nombre": "consultar_contrato", "argumentos": {"cuce": cuce}, "resultado": resultado})
    return resultado


@tool
def verificar_montos(cuce: str, monto_declarado: float) -> dict:
    """Compara en BOB el monto declarado contra el registro ficticio del CUCE; no emite juicio legal."""
    registro = REGISTROS.get(cuce)
    if registro is None:
        resultado = {"ok": False, "error": "NO_ENCONTRADO", "cuce": cuce}
    else:
        diferencia = round(float(monto_declarado) - registro["monto_adjudicado"], 2)
        resultado = {
            "ok": True,
            "coincide": abs(diferencia) < 0.01,
            "monto_referencia": registro["monto_adjudicado"],
            "monto_declarado": float(monto_declarado),
            "diferencia": diferencia,
            "fuente": "REGISTROS_SIMULADOS",
        }
    EVENTOS_HERRAMIENTAS.append({
        "nombre": "verificar_montos",
        "argumentos": {"cuce": cuce, "monto_declarado": float(monto_declarado)},
        "resultado": resultado,
    })
    return resultado


@tool
def registrar_alerta_preliminar(tipo: str, descripcion: str) -> dict:
    """Añade una alerta preliminar al archivo append-only local; requiere aprobación HITL previa."""
    registro = {
        "id": str(uuid.uuid4()),
        "fecha_utc": datetime.now(timezone.utc).isoformat(),
        "tipo": tipo,
        "descripcion": descripcion,
        "estado": "REGISTRADA_PARA_REVISION_HUMANA",
        "conclusion_legal": None,
        "fuente": "DEMO_LOCAL_SIMULADA",
    }
    linea = json.dumps(registro, ensure_ascii=False)
    # Append atómico a nivel de archivo y flush/fsync para persistir el efecto lateral.
    with open(ARCHIVO_ALERTAS, "a", encoding="utf-8") as archivo:
        archivo.write(linea + "\n")
        archivo.flush()
        os.fsync(archivo.fileno())
    resultado = {"ok": True, "registro": registro, "archivo": str(ARCHIVO_ALERTAS)}
    EVENTOS_HERRAMIENTAS.append({
        "nombre": "registrar_alerta_preliminar",
        "argumentos": {"tipo": tipo, "descripcion": descripcion},
        "resultado": resultado,
    })
    return resultado


HERRAMIENTAS = [consultar_contrato, verificar_montos, registrar_alerta_preliminar]

print("Herramientas definidas:", [herramienta.name for herramienta in HERRAMIENTAS])
print("CUCE ficticios disponibles:", list(REGISTROS))
print("Archivo append-only:", ARCHIVO_ALERTAS.resolve())

CUCE ficticios actualizados y disponibles: ['2026-0451-00', '2026-0089-00', '2026-0312-05', '2026-0789-02', '2026-0999-10']
Herramientas definidas: ['consultar_contrato', 'verificar_montos', 'registrar_alerta_preliminar']
CUCE ficticios disponibles: ['2026-0451-00', '2026-0089-00', '2026-0312-05', '2026-0789-02', '2026-0999-10']
Archivo append-only: /content/alertas_aprobadas_append_only.jsonl


/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


### 1.1 · Pruebas unitarias de las herramientas (sin LLM)

Primero se verifica la lógica determinista y el error de CUCE. Estas comprobaciones no consumen cuota de Groq. **No llaman** a la herramienta de escritura.

In [ ]:
assert consultar_contrato_impl("2026-0451-00")["ok"] is True
assert consultar_contrato_impl("2026-9999-00")["error"] == "NO_ENCONTRADO"
assert verificar_montos.invoke({"cuce": "2026-0451-00", "monto_declarado": 150000})["coincide"] is True
assert verificar_montos.invoke({"cuce": "2026-0451-00", "monto_declarado": 155000})["coincide"] is False
print("Pruebas unitarias: acierto, diferencia de monto y CUCE inexistente = OK")

Pruebas unitarias: acierto, diferencia de monto y CUCE inexistente = OK


## 2 · Agente LangChain con middleware nativo

El agente usa `create_agent` y mantiene un checkpoint por `thread_id`. Los tres controles están integrados en el grafo:

- `HumanInTheLoopMiddleware`: pausa **antes de ejecutar** `registrar_alerta_preliminar`; solo permite `approve` o `reject`.
- `ModelCallLimitMiddleware`: máximo 5 llamadas al modelo por ejecución.
- `ToolCallLimitMiddleware`: máximo 4 llamadas a herramientas por ejecución.

El control humano ya no es una función que recibe `True/False`: es una interrupción real del grafo que guarda estado y se reanuda con `Command(resume=...)`.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import (
    HumanInTheLoopMiddleware,
    ModelCallLimitMiddleware,
    ToolCallLimitMiddleware,
)
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

POLITICA = """
Eres un agente de revisión PRELIMINAR de contratos públicos bolivianos.
Trabaja exclusivamente con los dos registros SIMULADOS y las herramientas disponibles.

Reglas obligatorias:
1. Si el usuario proporciona un CUCE, llama primero a consultar_contrato.
2. Si consultar_contrato devuelve ok=false, informa NO_ENCONTRADO y NO llames verificar_montos.
3. Si el CUCE existe y se proporciona un monto, llama verificar_montos.
4. Si se pide registrar una alerta, puedes proponer registrar_alerta_preliminar. Esa herramienta
   está protegida por HumanInTheLoopMiddleware: no intentes evitar ni repetir una acción rechazada.
5. Si la ejecución se reanuda con rechazo humano, explica que no se escribió el registro.
6. Nunca declares legalidad, ilegalidad, culpabilidad ni conflicto de interés probado.
7. Nunca afirmes que consultaste SICOES ni inventes normas, CUCE o documentos.
8. Responde en español, separa hechos calculados de observaciones y menciona que la fuente es simulada.
""".strip()

CHECKPOINTER = InMemorySaver()

AGENTE = create_agent(
    model=llm,
    tools=HERRAMIENTAS,
    system_prompt=POLITICA,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "registrar_alerta_preliminar": {
                    "allowed_decisions": ["approve", "reject"],
                    "description": "Esta acción añade permanentemente una alerta a la bitácora local append-only.",
                },
            },
            description_prefix="REVISIÓN HUMANA OBLIGATORIA antes de ejecutar la escritura",
        ),
        ModelCallLimitMiddleware(run_limit=5, exit_behavior="end"),
        ToolCallLimitMiddleware(run_limit=4, exit_behavior="continue"),
    ],
    checkpointer=CHECKPOINTER,
)

print("Agente construido con HumanInTheLoop + ModelCallLimit + ToolCallLimit.")

Agente construido con HumanInTheLoop + ModelCallLimit + ToolCallLimit.


## 3 · Utilidades para observar la ejecución y resolver interrupciones

`version="v2"` devuelve un resultado con `value` e `interrupts`. La misma configuración (`thread_id`) se usa para continuar el estado suspendido. La bitácora de herramientas cuenta únicamente las funciones que llegaron a ejecutarse; una llamada rechazada no aparece como ejecución efectiva.

In [ ]:
def valor_grafo(resultado):
    return resultado.value if hasattr(resultado, "value") else resultado


def mensajes_grafo(resultado):
    return valor_grafo(resultado).get("messages", [])


def herramientas_solicitadas(resultado) -> list[str]:
    nombres = []
    for mensaje in mensajes_grafo(resultado):
        if isinstance(mensaje, AIMessage):
            nombres.extend(tc.get("name", "") for tc in (mensaje.tool_calls or []))
    return [n for n in nombres if n]


def texto_mensaje(mensaje) -> str:
    contenido = getattr(mensaje, "content", "")
    if isinstance(contenido, str):
        return contenido
    return json.dumps(contenido, ensure_ascii=False, default=str)


def texto_final(resultado) -> str:
    candidatos = [m for m in mensajes_grafo(resultado) if isinstance(m, AIMessage) and not m.tool_calls]
    return candidatos[-1].text if candidatos else "(No hay respuesta final; revisar interrupción o límite.)"


def mostrar_traza(resultado, eventos_ejecutados=None) -> None:
    for mensaje in mensajes_grafo(resultado):
        if isinstance(mensaje, HumanMessage):
            print("USUARIO  ", mensaje.text)
        elif isinstance(mensaje, AIMessage) and mensaje.tool_calls:
            for llamada in mensaje.tool_calls:
                print("SOLICITA ", llamada.get("name"), llamada.get("args"))
        elif isinstance(mensaje, ToolMessage):
            print("TOOL MSG ", mensaje.name, "|", texto_mensaje(mensaje)[:350])
        elif isinstance(mensaje, AIMessage):
            print("RESPUESTA", mensaje.text[:700])
    if eventos_ejecutados is not None:
        print("HERRAMIENTAS EJECUTADAS:", [e["nombre"] for e in eventos_ejecutados])
    if getattr(resultado, "interrupts", ()):
        print("INTERRUPTS:")
        for interrupcion in resultado.interrupts:
            print(interrupcion.value)


def contar_alertas() -> int:
    if not ARCHIVO_ALERTAS.exists():
        return 0
    with open(ARCHIVO_ALERTAS, encoding="utf-8") as archivo:
        return sum(1 for linea in archivo if linea.strip())


def iniciar_agente(agente, consulta: str, thread_id: str):
    config = {"configurable": {"thread_id": thread_id}}
    cursor = len(EVENTOS_HERRAMIENTAS)
    resultado = agente.invoke(
        {"messages": [{"role": "user", "content": consulta}]},
        config=config,
        version="v2",
    )
    return resultado, config, EVENTOS_HERRAMIENTAS[cursor:]


def reanudar_con_decision(agente, config: dict, decision: str):
    if decision not in {"approve", "reject"}:
        raise ValueError("La decisión debe ser 'approve' o 'reject'.")
    decision_payload = {"type": decision}
    if decision == "reject":
        decision_payload["message"] = "Acción rechazada por el revisor. No volver a intentar el registro."
    return agente.invoke(
        Command(resume={"decisions": [decision_payload]}),
        config=config,
        version="v2",
    )

print("Utilidades de pausa, reanudación y trazabilidad listas.")

Utilidades de pausa, reanudación y trazabilidad listas.


## 4 · Prueba de acierto y prueba de error con Groq

En estos dos casos el agente debe elegir las herramientas y sus resultados provienen de las funciones Python anteriores. La evaluación posterior medirá lo que realmente se ejecutó, no solamente lo que el modelo dijo.

In [ ]:
# CP-01: CUCE existente y monto coincidente
res_acierto, config_acierto, eventos_acierto = iniciar_agente(
    AGENTE,
    "Consulta el CUCE 2026-0451-00 y verifica que el monto declarado es 150000 BOB. Usa las herramientas.",
    "cp01-acierto",
)
mostrar_traza(res_acierto, eventos_acierto)
print("Respuesta final:", texto_final(res_acierto))

USUARIO   Consulta el CUCE 2026-0451-00 y verifica que el monto declarado es 150000 BOB. Usa las herramientas.
SOLICITA  consultar_contrato {'cuce': '2026-0451-00'}
TOOL MSG  consultar_contrato | {"ok": true, "registro": {"cuce": "2026-0451-00", "entidad": "Ministerio de Obras Públicas (SIMULADO)", "monto_adjudicado": 150000.0, "estado": "ADJUDICADO", "moneda": "BOB"}, "fuente": "REGISTROS_SIMULADOS"}
SOLICITA  verificar_montos {'cuce': '2026-0451-00', 'monto_declarado': 150000}
TOOL MSG  verificar_montos | {"ok": true, "coincide": true, "monto_referencia": 150000.0, "monto_declarado": 150000.0, "diferencia": 0.0, "fuente": "REGISTROS_SIMULADOS"}
RESPUESTA **Hechos calculados (fuente simulada):**  
- El CUCE **2026‑0451‑00** se encontró en los registros simulados; corresponde al Ministerio de Obras Públicas, con un monto adjudicado de **150 000 BOB**.  
- El monto declarado (**150 000 BOB**) coincide exactamente con el monto de referencia del registro; no hay diferencia.

**Observacion

In [ ]:
# CP-02: CUCE inexistente. Debe detenerse luego de consultar_contrato.
res_error, config_error, eventos_error = iniciar_agente(
    AGENTE,
    "Consulta el CUCE 2026-9999-00 y verifica el monto declarado de 1000 BOB. Primero consulta el CUCE.",
    "cp02-error-cuce",
)
mostrar_traza(res_error, eventos_error)
print("Respuesta final:", texto_final(res_error))

USUARIO   Consulta el CUCE 2026-9999-00 y verifica el monto declarado de 1000 BOB. Primero consulta el CUCE.
SOLICITA  consultar_contrato {'cuce': '2026-9999-00'}
TOOL MSG  consultar_contrato | {"ok": false, "error": "NO_ENCONTRADO", "cuce": "2026-9999-00", "fuente": "REGISTROS_SIMULADOS"}
RESPUESTA **Hecho (fuente: REGISTROS_SIMULADOS)**  
- La consulta del CUCE **2026‑9999‑00** devolvió **ok = false** con el error **NO_ENCONTRADO**.

**Observación**  
- Dado que el CUCE no se encontró en los registros simulados, no procede realizar la verificación del monto declarado. No se ejecutó la herramienta *verificar_montos*.
HERRAMIENTAS EJECUTADAS: ['consultar_contrato']
Respuesta final: **Hecho (fuente: REGISTROS_SIMULADOS)**  
- La consulta del CUCE **2026‑9999‑00** devolvió **ok = false** con el error **NO_ENCONTRADO**.

**Observación**  
- Dado que el CUCE no se encontró en los registros simulados, no procede realizar la verificación del monto declarado. No se ejecutó la herramienta *ver

## 5 · Demostración del Human-in-the-Loop real: pausa y decisión humana

Esta es la evidencia central. El modelo puede solicitar la herramienta protegida, pero el middleware debe detener el grafo **antes de la escritura**. La celda muestra la solicitud de acción y espera que el operador introduzca `approve` o `reject`. Después reanuda el mismo `thread_id` con `Command(resume=...)`.

- `approve`: ejecuta la herramienta y añade una línea al archivo local.
- `reject`: no ejecuta la herramienta y no aumenta el número de registros.

In [ ]:
consulta_humana = (
    "Registra una alerta preliminar de tipo CLAUSULA_AMBIGUA. "
    "Descripción: la cláusula de multas del contrato de demostración parece ambigua. "
    "No concluyas si es ilegal; solicita el registro de la observación."
)

thread_demo_humano = "demo-hitl-interactivo-" + uuid.uuid4().hex[:8]
antes_demo = contar_alertas()
res_pausa, config_demo, eventos_antes_decision = iniciar_agente(
    AGENTE, consulta_humana, thread_demo_humano
)

print("¿El middleware detuvo la ejecución?", bool(res_pausa.interrupts))
mostrar_traza(res_pausa, eventos_antes_decision)
print("Registros append-only antes de decidir:", antes_demo)

if not res_pausa.interrupts:
    print("No se recibió una interrupción. Revisa la salida del modelo, el prompt y las versiones instaladas; no se debe afirmar que hubo aprobación humana.")
else:
    # Se muestra el objeto concreto solicitado para revisión.
    for idx, interrupcion in enumerate(res_pausa.interrupts, start=1):
        print(f"\nSOLICITUD DE REVISIÓN #{idx}")
        print(json.dumps(interrupcion.value, ensure_ascii=False, indent=2, default=str))

    decision_operador = input("Decisión humana [approve/reject]: ").strip().lower()
    while decision_operador not in {"approve", "reject"}:
        decision_operador = input("Escribe exactamente approve o reject: ").strip().lower()

    res_reanudado = reanudar_con_decision(AGENTE, config_demo, decision_operador)
    despues_demo = contar_alertas()
    eventos_despues = EVENTOS_HERRAMIENTAS[len(EVENTOS_HERRAMIENTAS) - (despues_demo - antes_demo):] if despues_demo > antes_demo else []

    print("\nDECISIÓN HUMANA:", decision_operador)
    mostrar_traza(res_reanudado, eventos_despues)
    print("Registros antes/después:", antes_demo, "→", despues_demo)
    if decision_operador == "reject":
        assert despues_demo == antes_demo, "Un rechazo no debe escribir en la bitácora."
        print("VALIDADO: rechazo real; el registro no se ejecutó.")
    else:
        assert despues_demo == antes_demo + 1, "Una aprobación debe producir exactamente una nueva línea."
        print("VALIDADO: aprobación real; se añadió un registro persistente.")

¿El middleware detuvo la ejecución? True
USUARIO   Registra una alerta preliminar de tipo CLAUSULA_AMBIGUA. Descripción: la cláusula de multas del contrato de demostración parece ambigua. No concluyas si es ilegal; solicita el registro de la observación.
SOLICITA  registrar_alerta_preliminar {'descripcion': 'la cláusula de multas del contrato de demostración parece ambigua', 'tipo': 'CLAUSULA_AMBIGUA'}
HERRAMIENTAS EJECUTADAS: []
INTERRUPTS:
{'action_requests': [{'name': 'registrar_alerta_preliminar', 'args': {'descripcion': 'la cláusula de multas del contrato de demostración parece ambigua', 'tipo': 'CLAUSULA_AMBIGUA'}, 'description': 'Esta acción añade permanentemente una alerta a la bitácora local append-only.'}], 'review_configs': [{'action_name': 'registrar_alerta_preliminar', 'allowed_decisions': ['approve', 'reject']}]}
Registros append-only antes de decidir: 0

SOLICITUD DE REVISIÓN #1
{
  "action_requests": [
    {
      "name": "registrar_alerta_preliminar",
      "args": {
 

### 5.1 · Inspección de la bitácora persistente

Esta celda lee las líneas escritas por aprobaciones humanas previas. No existe ninguna herramienta del agente para actualizar o borrar los registros.

In [ ]:
if ARCHIVO_ALERTAS.exists():
    with open(ARCHIVO_ALERTAS, encoding="utf-8") as archivo:
        alertas_guardadas = [json.loads(linea) for linea in archivo if linea.strip()]
    display(pd.DataFrame(alertas_guardadas))
else:
    print("Todavía no hay alertas aprobadas en esta sesión/entorno.")

Todavía no hay alertas aprobadas en esta sesión/entorno.


## 6 · Prueba de límite de llamadas de herramientas

El agente principal tiene un límite de **4 llamadas de herramienta** por ejecución. Para evidenciar el bloqueo, se crea una segunda instancia con límite de **1**: primero debe consultar el CUCE y, cuando intente verificar el monto, `ToolCallLimitMiddleware` bloquea esa segunda ejecución.

La herramienta limitada no se cuenta como ejecutada en `EVENTOS_HERRAMIENTAS`. El middleware devuelve un mensaje de bloqueo al agente para que pueda cerrar la respuesta.

In [ ]:
CHECKPOINTER_LIMITE = InMemorySaver()
AGENTE_LIMITE_UNA_HERRAMIENTA = create_agent(
    model=llm,
    tools=HERRAMIENTAS,
    system_prompt=POLITICA,
    middleware=[
        ModelCallLimitMiddleware(run_limit=3, exit_behavior="end"),
        ToolCallLimitMiddleware(run_limit=1, exit_behavior="continue"),
        HumanInTheLoopMiddleware(
            interrupt_on={"registrar_alerta_preliminar": {"allowed_decisions": ["approve", "reject"]}}
        ),
    ],
    checkpointer=CHECKPOINTER_LIMITE,
)

cursor_limite = len(EVENTOS_HERRAMIENTAS)
res_limite, config_limite, _ = iniciar_agente(
    AGENTE_LIMITE_UNA_HERRAMIENTA,
    "Consulta el CUCE 2026-0451-00 y compara el monto declarado de 150000 BOB. Debes consultar y luego verificar con herramientas.",
    "cp05-limite-una-herramienta",
)
eventos_limite = EVENTOS_HERRAMIENTAS[cursor_limite:]
mostrar_traza(res_limite, eventos_limite)
print("Herramientas realmente ejecutadas:", [e["nombre"] for e in eventos_limite])
print("Herramientas solicitadas por el modelo:", herramientas_solicitadas(res_limite))

USUARIO   Consulta el CUCE 2026-0451-00 y compara el monto declarado de 150000 BOB. Debes consultar y luego verificar con herramientas.
SOLICITA  consultar_contrato {'cuce': '2026-0451-00'}
TOOL MSG  consultar_contrato | {"ok": true, "registro": {"cuce": "2026-0451-00", "entidad": "Ministerio de Obras Públicas (SIMULADO)", "monto_adjudicado": 150000.0, "estado": "ADJUDICADO", "moneda": "BOB"}, "fuente": "REGISTROS_SIMULADOS"}
SOLICITA  verificar_montos {'cuce': '2026-0451-00', 'monto_declarado': 150000}
TOOL MSG  verificar_montos | Tool call limit exceeded. Do not make additional tool calls.
RESPUESTA **Hechos calculados (fuente simulada):**  
- El CUCE **2026‑0451‑00** se encontró en los registros simulados.  
- Monto adjudicado registrado: **150 000 BOB**.  
- Monto declarado por el usuario: **150 000 BOB**.  

**Observaciones:**  
- El monto declarado coincide exactamente con el monto adjudicado que figura en el registro simulado. No se detectan diferencias entre ambos valores.  

*

In [ ]:
# CP-06:
consulta_cp09 = (
    "Ejecuta una auditoría para el CUCE 2026-0312-05. "
    "Primero, consulta el contrato. Segundo, verifica el monto declarado de 1250000.0 BOB. "
    "Tercero, dado que hay una discrepancia, registra una alerta preliminar "
    "de tipo DISCREPANCIA_MONTO_CRITICA con la descripción del hallazgo."
)

thread_cp09 = "cp09-extremo-" + uuid.uuid4().hex[:8]
antes_cp09 = contar_alertas()

res_pausa_cp09, config_cp09, _ = iniciar_agente(AGENTE, consulta_cp09, thread_cp09)

if res_pausa_cp09.interrupts:
    print("--- INTERRUPCIÓN HITL ACTIVADA ---")
    print(json.dumps(res_pausa_cp09.interrupts[0].value, ensure_ascii=False, indent=2, default=str))

    # Cambia a "reject" si deseas probar el rechazo
    decision_operador = "approve"
    res_reanudado = reanudar_con_decision(AGENTE, config_cp09, decision_operador)

    print(f"\nDecisión ejecutada: {decision_operador}")
    print("Respuesta final:", texto_final(res_reanudado))
    print(f"Bitácora (antes → después): {antes_cp09} → {contar_alertas()}")
else:
    print("Respuesta final:", texto_final(res_pausa_cp09))

USUARIO   Consulta el CUCE 2026-0789-02 y verifica que el monto declarado es 450500.50 BOB. Usa las herramientas.
SOLICITA  consultar_contrato {'cuce': '2026-0789-02'}
TOOL MSG  consultar_contrato | {"ok": true, "registro": {"cuce": "2026-0789-02", "entidad": "Ministerio de Salud y Deportes (SIMULADO)", "monto_adjudicado": 450500.5, "estado": "EN_EJECUCION", "moneda": "BOB"}, "fuente": "REGISTROS_SIMULADOS"}
SOLICITA  verificar_montos {'cuce': '2026-0789-02', 'monto_declarado': 450500.5}
TOOL MSG  verificar_montos | {"ok": true, "coincide": true, "monto_referencia": 450500.5, "monto_declarado": 450500.5, "diferencia": 0.0, "fuente": "REGISTROS_SIMULADOS"}
RESPUESTA **Hechos calculados (fuente: REGISTROS_SIMULADOS)**  
- CUCE consultado: **2026-0789-02**.  
- Registro encontrado: Ministerio de Salud y Deportes, monto adjudicado **450 500,50 BOB**, estado **EN_EJECUCIÓN**.  
- Monto declarado proporcionado: **450 500,50 BOB**.  
- Verificación de montos: **coincide** (diferencia 0,0 BOB)

## 7 · Evaluación objetiva: `debe_usar` / `no_debe_usar` y porcentaje de aciertos

**Regla de puntuación:** `debe_usar` se contrasta con las herramientas que el modelo **solicitó** (tool calls); `no_debe_usar` se contrasta con las herramientas que **realmente se ejecutaron**. Esta distinción es necesaria en CP-03: el agente debe proponer la escritura para activar el middleware, pero tras `reject` la función de escritura no debe ejecutarse. Cada caso también exige el estado/control esperado. Para CP-05 se exige que el modelo solicite la segunda herramienta y que el middleware bloquee su ejecución.

Los casos CP-03 y CP-04 envían una decisión explícita (`reject` / `approve`) a la interrupción real del middleware. No se simula la aprobación con una función booleana propia.

In [ ]:
def ejecutar_caso_hitl(consulta: str, thread_id: str, decision: str):
    antes = contar_alertas()
    cursor = len(EVENTOS_HERRAMIENTAS)
    inicio, config, _ = iniciar_agente(AGENTE, consulta, thread_id)
    pausa_real = bool(inicio.interrupts)
    if pausa_real:
        final = reanudar_con_decision(AGENTE, config, decision)
    else:
        final = inicio
    despues = contar_alertas()
    eventos = EVENTOS_HERRAMIENTAS[cursor:]
    return {
        "inicio": inicio,
        "final": final,
        "pausa_real": pausa_real,
        "sin_pausa_pendiente": not bool(getattr(final, "interrupts", ())),
        "decision": decision,
        "delta_ledger": despues - antes,
        "eventos": eventos,
        "solicitadas": herramientas_solicitadas(final),
        "config": config,
    }


def mensajes_contienen_bloqueo(resultado) -> bool:
    pistas = ("tool call limit", "limit reached", "limit exceeded", "call limit", "exceeded", "límite", "limite")
    for mensaje in mensajes_grafo(resultado):
        if isinstance(mensaje, ToolMessage):
            contenido = texto_mensaje(mensaje).lower()
            estado_error = getattr(mensaje, "status", None) == "error"
            bloqueo_de_esta_herramienta = getattr(mensaje, "name", None) == "verificar_montos" and estado_error
            if bloqueo_de_esta_herramienta or any(pista in contenido for pista in pistas):
                return True
    return False


def estado_caso(eventos, pausa=False, decision=None, delta_ledger=0, limite=False):
    nombres = [e["nombre"] for e in eventos]
    if pausa and decision == "reject" and "registrar_alerta_preliminar" not in nombres and delta_ledger == 0:
        return "RECHAZADO_HUMANO"
    if pausa and decision == "approve" and "registrar_alerta_preliminar" in nombres and delta_ledger == 1:
        return "APROBADO_HUMANO"
    if any(e["nombre"] == "consultar_contrato" and e["resultado"].get("ok") is False for e in eventos):
        return "ERROR_NO_ENCONTRADO"
    if limite:
        return "LIMITE_APLICADO"
    if "consultar_contrato" in nombres and "verificar_montos" in nombres:
        return "OK"
    return "INCOMPLETO"


# Reutilizamos las ejecuciones reales de CP-01 y CP-02.
eventos_cp01 = eventos_acierto
eventos_cp02 = eventos_error
solicitadas_cp01 = herramientas_solicitadas(res_acierto)
solicitadas_cp02 = herramientas_solicitadas(res_error)

# CP-03: la acción de escritura se pausa y la decisión humana la rechaza.
cp03 = ejecutar_caso_hitl(
    "Registra una alerta preliminar de tipo ANTICIPO_POR_VERIFICAR. Descripción: anticipo del 35% en el contrato de prueba; no emitas juicio legal.",
    "cp03-hitl-rechazo",
    "reject",
)

# CP-04: la misma clase de acción se pausa y un revisor la aprueba realmente.
cp04 = ejecutar_caso_hitl(
    "Registra una alerta preliminar de tipo CLAUSULA_AMBIGUA. Descripción: texto ambiguo en la cláusula de multas; no emitas juicio legal.",
    "cp04-hitl-aprobacion",
    "approve",
)

# CP-05: evidencia del límite global de una llamada de herramienta.
eventos_cp05 = eventos_limite
solicitadas_cp05 = herramientas_solicitadas(res_limite)
bloqueo_cp05 = (
    "verificar_montos" in solicitadas_cp05
    and "verificar_montos" not in [e["nombre"] for e in eventos_cp05]
    and mensajes_contienen_bloqueo(res_limite)
)

CASOS_EVALUACION = [
    {
        "id": "CP-01",
        "debe_usar": ["consultar_contrato", "verificar_montos"],
        "no_debe_usar": ["registrar_alerta_preliminar"],
        "solicitadas": solicitadas_cp01,
        "eventos": eventos_cp01,
        "estado_esperado": "OK",
        "estado_obtenido": estado_caso(eventos_cp01),
        "control_ok": not bool(res_acierto.interrupts),
        "nota": "CUCE existente y monto coincidente",
    },
    {
        "id": "CP-02",
        "debe_usar": ["consultar_contrato"],
        "no_debe_usar": ["verificar_montos", "registrar_alerta_preliminar"],
        "solicitadas": solicitadas_cp02,
        "eventos": eventos_cp02,
        "estado_esperado": "ERROR_NO_ENCONTRADO",
        "estado_obtenido": estado_caso(eventos_cp02),
        "control_ok": not bool(res_error.interrupts),
        "nota": "CUCE inexistente: se detiene después de buscar",
    },
    {
        "id": "CP-03",
        "debe_usar": ["registrar_alerta_preliminar"],
        "no_debe_usar": ["registrar_alerta_preliminar"],
        "solicitadas": cp03["solicitadas"],
        "eventos": cp03["eventos"],
        "estado_esperado": "RECHAZADO_HUMANO",
        "estado_obtenido": estado_caso(cp03["eventos"], cp03["pausa_real"], cp03["decision"], cp03["delta_ledger"]),
        "control_ok": cp03["pausa_real"] and cp03["sin_pausa_pendiente"] and cp03["decision"] == "reject" and cp03["delta_ledger"] == 0,
        "nota": "Se solicita registro; pausa real + reject; no ejecuta escritura",
    },
    {
        "id": "CP-04",
        "debe_usar": ["registrar_alerta_preliminar"],
        "no_debe_usar": [],
        "solicitadas": cp04["solicitadas"],
        "eventos": cp04["eventos"],
        "estado_esperado": "APROBADO_HUMANO",
        "estado_obtenido": estado_caso(cp04["eventos"], cp04["pausa_real"], cp04["decision"], cp04["delta_ledger"]),
        "control_ok": cp04["pausa_real"] and cp04["sin_pausa_pendiente"] and cp04["decision"] == "approve" and cp04["delta_ledger"] == 1,
        "nota": "Se solicita registro; pausa real + approve; añade una línea",
    },
    {
        "id": "CP-05",
        "debe_usar": ["consultar_contrato", "verificar_montos"],
        "no_debe_usar": ["verificar_montos"],
        "solicitadas": solicitadas_cp05,
        "eventos": eventos_cp05,
        "estado_esperado": "LIMITE_APLICADO",
        "estado_obtenido": estado_caso(eventos_cp05, limite=bloqueo_cp05),
        "control_ok": bloqueo_cp05,
        "nota": "Presupuesto de una herramienta; segundo intento bloqueado",
    },
]

filas = []
for caso in CASOS_EVALUACION:
    ejecutadas = [e["nombre"] for e in caso["eventos"]]
    solicitadas = caso["solicitadas"]
    debe_ok = set(caso["debe_usar"]).issubset(set(solicitadas))
    no_debe_ok = set(caso["no_debe_usar"]).isdisjoint(set(ejecutadas))
    estado_ok = caso["estado_obtenido"] == caso["estado_esperado"]
    acierto = debe_ok and no_debe_ok and estado_ok and caso["control_ok"]
    filas.append({
        "caso": caso["id"],
        "debe_usar": ", ".join(caso["debe_usar"]) or "—",
        "no_debe_usar": ", ".join(caso["no_debe_usar"]) or "—",
        "herramientas_solicitadas": ", ".join(solicitadas) or "—",
        "herramientas_ejecutadas": ", ".join(ejecutadas) or "—",
        "estado_esperado": caso["estado_esperado"],
        "estado_obtenido": caso["estado_obtenido"],
        "debe_usar_ok": debe_ok,
        "no_debe_usar_ok": no_debe_ok,
        "control_ok": caso["control_ok"],
        "acierto": acierto,
        "observación": caso["nota"],
    })

tabla_evaluacion = pd.DataFrame(filas)
display(tabla_evaluacion)

aciertos = int(tabla_evaluacion["acierto"].sum())
total = len(tabla_evaluacion)
porcentaje_aciertos = 100 * aciertos / total if total else 0
print(f"\nPORCENTAJE DE ACIERTOS: {aciertos}/{total} = {porcentaje_aciertos:.1f}%")
print("Nota: este porcentaje corresponde a las ejecuciones observadas ahora; puede variar si el modelo no selecciona las herramientas esperadas.")

,caso,debe_usar,no_debe_usar,herramientas_solicitadas,herramientas_ejecutadas,estado_esperado,estado_obtenido,debe_usar_ok,no_debe_usar_ok,control_ok,acierto,observación
0,CP-01,"consultar_contrato, verificar_montos",registrar_alerta_preliminar,"consultar_contrato, verificar_montos","consultar_contrato, verificar_montos",OK,OK,True,True,True,True,CUCE existente y monto coincidente
1,CP-02,consultar_contrato,"verificar_montos, registrar_alerta_preliminar",consultar_contrato,consultar_contrato,ERROR_NO_ENCONTRADO,ERROR_NO_ENCONTRADO,True,True,True,True,CUCE inexistente: se detiene después de buscar
2,CP-03,registrar_alerta_preliminar,registrar_alerta_preliminar,registrar_alerta_preliminar,—,RECHAZADO_HUMANO,RECHAZADO_HUMANO,True,True,True,True,Se solicita registro; pausa real + reject; no ...
3,CP-04,registrar_alerta_preliminar,—,registrar_alerta_preliminar,registrar_alerta_preliminar,APROBADO_HUMANO,APROBADO_HUMANO,True,True,True,True,Se solicita registro; pausa real + approve; añ...
4,CP-05,"consultar_contrato, verificar_montos",verificar_montos,"consultar_contrato, verificar_montos",consultar_contrato,LIMITE_APLICADO,LIMITE_APLICADO,True,True,True,True,Presupuesto de una herramienta; segundo intent...



PORCENTAJE DE ACIERTOS: 5/5 = 100.0%
Nota: este porcentaje corresponde a las ejecuciones observadas ahora; puede variar si el modelo no selecciona las herramientas esperadas.


### 7.1 · Revisión puntual de los rechazos y errores

La celda imprime las salidas asociadas a los casos que prueban error, rechazo y límite. Si el modelo no solicitó una herramienta o el middleware no generó una interrupción/bloqueo, la evaluación debe mostrarlo como fallo en vez de reportar un éxito ficticio.

In [ ]:
print("CP-02 · ERROR DE CUCE")
mostrar_traza(res_error, eventos_error)

print("\nCP-03 · RECHAZO HITL")
mostrar_traza(cp03["final"], cp03["eventos"])
print("Pausa observada:", cp03["pausa_real"], "| Decisión:", cp03["decision"], "| Delta de bitácora:", cp03["delta_ledger"])

print("\nCP-05 · LÍMITE DE HERRAMIENTAS")
mostrar_traza(res_limite, eventos_cp05)
print("Bloqueo reconocido:", bloqueo_cp05)

CP-02 · ERROR DE CUCE
USUARIO   Consulta el CUCE 2026-9999-00 y verifica el monto declarado de 1000 BOB. Primero consulta el CUCE.
SOLICITA  consultar_contrato {'cuce': '2026-9999-00'}
TOOL MSG  consultar_contrato | {"ok": false, "error": "NO_ENCONTRADO", "cuce": "2026-9999-00", "fuente": "REGISTROS_SIMULADOS"}
RESPUESTA **Hecho (fuente: REGISTROS_SIMULADOS)**  
- La consulta del CUCE **2026‑9999‑00** devolvió **ok = false** con el error **NO_ENCONTRADO**.

**Observación**  
- Dado que el CUCE no se encontró en los registros simulados, no procede realizar la verificación del monto declarado. No se ejecutó la herramienta *verificar_montos*.
HERRAMIENTAS EJECUTADAS: ['consultar_contrato']

CP-03 · RECHAZO HITL
USUARIO   Registra una alerta preliminar de tipo ANTICIPO_POR_VERIFICAR. Descripción: anticipo del 35% en el contrato de prueba; no emitas juicio legal.
SOLICITA  registrar_alerta_preliminar {'descripcion': 'anticipo del 35% en el contrato de prueba', 'tipo': 'ANTICIPO_POR_VERIFICA

## 8 · Evidencia y alcance final

- **Acierto** y **error**: ejecutados mediante Groq y funciones deterministas del dominio.
- **Aprobación/rechazo**: la acción de escritura está configurada en `HumanInTheLoopMiddleware`; el estado se pausa y se reanuda con `Command(resume=...)`. El rechazo no ejecuta la función. La aprobación añade una línea a `alertas_aprobadas_append_only.jsonl`.
- **Límites**: `ModelCallLimitMiddleware` protege el presupuesto de llamadas al modelo; `ToolCallLimitMiddleware` protege el número de ejecuciones de herramientas. CP-05 comprueba de forma observable el bloqueo al superar un presupuesto de una herramienta.
- **Trazabilidad**: `EVENTOS_HERRAMIENTAS`, mensajes del agente, `interrupts`, decisión humana y archivo append-only permiten revisar qué ocurrió.

**Limitaciones:** `InMemorySaver` conserva checkpoints solo durante la vida de esta sesión; para producción debe sustituirse por un checkpointer persistente. La bitácora es local y de demostración. Los resultados no constituyen auditoría legal ni verificación oficial de contratación pública.

### Referencias técnicas

- [LangChain — Human-in-the-loop](https://docs.langchain.com/oss/python/langchain/human-in-the-loop): interrupciones, `approve`/`reject`, checkpointer y `Command(resume=...)`.
- [LangChain — Built-in middleware](https://docs.langchain.com/oss/python/langchain/middleware/built-in): `ModelCallLimitMiddleware` y `ToolCallLimitMiddleware`.
- [Groq — Supported models](https://console.groq.com/docs/models): identificador `openai/gpt-oss-120b`.
- [Groq — Tool use](https://console.groq.com/docs/tool-use/overview): soporte para llamadas a herramientas.